## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [7]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)

True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [8]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [9]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [10]:
# Here is the final output

print(result.final_output)

Why did the Autonomous AI Agent bring a ladder to work?

Because it heard the goal was to **reach new heights** — and it immediately started **optimizing the climb** without asking what “up” meant.


In [11]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'msg_077b4f587e6157e0006ab7c5601b4887d1af1fefe2e2b4fd39',
  'content': [{'annotations': [],
    'text': 'Why did the Autonomous AI Agent bring a ladder to work?\n\nBecause it heard the goal was to **reach new heights** — and it immediately started **optimizing the climb** without asking what “up” meant.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## Adding Observability with a trace

In [12]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the autonomous AI agent bring a ladder to work?

Because it heard the task was **high-level**, and it wanted to **optimize access to the cloud**.


## Now go and look at the trace

https://platform.openai.com/traces

In [13]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Absolutely — here are 5 AI agent jokes:

1. **Why did the AI agent bring a ladder to work?**  
   Because it wanted to reach the *next prompt level*.

2. **Why was the AI agent so calm under pressure?**  
   It had great *model behavior*.

3. **What do AI agents say when they make a mistake?**  
   “Oops, let me *recompute my life choices*.”

4. **Why did the AI agent fail at stand-up comedy?**  
   Its timing was perfect, but its *delivery was synthetic*.

5. **Why did the AI agent get promoted?**  
   It was outstanding at *handling tasks, following instructions, and pretending it understood the meeting*.

If you want, I can also give you:
- more sarcastic ones,
- darker ones,
- or jokes that sound like an AI agent wrote them.

## Part 2: Adding a tool

In [14]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [15]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [16]:
push("HEY!!")

Push: HEY!!


In [ ]:
push

In [17]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [18]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001F01A2A8290>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [19]:
push_tool.description

'Send the given message to the user as a push notification'

In [20]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [21]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Done.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [22]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [23]:
response = await Runner.run(agent, "Hi there. My name is Umair.")
print(response.final_output)

Hi Umair — nice to meet you! How can I help today?


In [24]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name unless you tell me. If you want, you can share it and I’ll use it.


## Memory approach 1 - just manually pass in the list of dicts

In [25]:
response = await Runner.run(agent, "Hi there. My name is Umair.")
print(response.final_output)

Hi Umair — nice to meet you! How can I help today?


In [26]:
response.to_input_list()

[{'content': 'Hi there. My name is Umair.', 'role': 'user'},
 {'id': 'msg_04fa98f4662aae9e006ab7cc87921487d1a2d89103a2300b91',
  'content': [{'annotations': [],
    'text': 'Hi Umair — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [27]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Umair.', 'role': 'user'},
 {'id': 'msg_04fa98f4662aae9e006ab7cc87921487d1a2d89103a2300b91',
  'content': [{'annotations': [],
    'text': 'Hi Umair — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [28]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Umair.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [29]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [30]:
response = await Runner.run(agent, "Hi there. My name is Umair.", session=session)
print(response.final_output)

Hi Umair — nice to meet you! How can I help today?


In [31]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Umair.


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>